### Text To Sql Agent

In [150]:
from langchain_community.utilities import SQLDatabase
from sqlalchemy import create_engine
import os 
from dotenv import load_dotenv

In [151]:
load_dotenv()

True

In [152]:
DB_USER=os.getenv("DB_USER")
DB_NAME=os.getenv("DB_NAME")
DB_PASSWORD=os.getenv("DB_PASSWORD")
DB_PORT=os.getenv("DB_PORT")
DB_HOST=os.getenv("DB_HOST")

In [153]:
from sqlalchemy.engine import URL

connection_url=URL.create(
    "mysql+pymysql",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME
)

In [154]:
engine=create_engine(
    connection_url
)

In [155]:
db=SQLDatabase(
    engine,
    sample_rows_in_table_info=0,
    ignore_tables=["_demodb_import"]
)

In [156]:
def get_schema():
    return db.get_table_info()

In [157]:
schema=get_schema()

In [158]:
print(schema)


CREATE TABLE album (
	`AlbumId` BIGINT NOT NULL AUTO_INCREMENT, 
	`Title` VARCHAR(160) NOT NULL, 
	`ArtistId` BIGINT NOT NULL, 
	PRIMARY KEY (`AlbumId`), 
	CONSTRAINT album_ibfk_1 FOREIGN KEY(`ArtistId`) REFERENCES artist (`ArtistId`)
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci


CREATE TABLE artist (
	`ArtistId` BIGINT NOT NULL AUTO_INCREMENT, 
	`Name` VARCHAR(120), 
	PRIMARY KEY (`ArtistId`)
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci


CREATE TABLE customer (
	`CustomerId` BIGINT NOT NULL AUTO_INCREMENT, 
	`FirstName` VARCHAR(40) NOT NULL, 
	`LastName` VARCHAR(20) NOT NULL, 
	`Company` VARCHAR(80), 
	`Address` VARCHAR(70), 
	`City` VARCHAR(40), 
	`State` VARCHAR(40), 
	`Country` VARCHAR(40), 
	`PostalCode` VARCHAR(10), 
	`Phone` VARCHAR(24), 
	`Fax` VARCHAR(24), 
	`Email` VARCHAR(60) NOT NULL, 
	`SupportRepId` BIGINT, 
	PRIMARY KEY (`CustomerId`), 
	CONSTRAINT customer_ibfk_1 FOREIGN KEY(`SupportRepId`) REFERENCES employee (`EmployeeId`)


In [159]:
def run_query(query):
    print(f"Query being executed:{query}")
    return db.run(query)

In [160]:
run_query("""
SELECT Artist.Name
FROM Artist
LIMIT 5;
""")

Query being executed:
SELECT Artist.Name
FROM Artist
LIMIT 5;



"[('AC/DC',), ('Accept',), ('Aerosmith',), ('Alanis Morissette',), ('Alice In Chains',)]"

### LLM Initilization

In [161]:
from langchain_groq import ChatGroq

GROQ_API_KEY=os.getenv("GROQ_API_KEY")

In [162]:
llm=ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=GROQ_API_KEY
)

### SQL Generator

In [163]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def write_sql_query(llm):

    template="""
        Based on the table schema below , write a SQL Query
        that would answer the user's question.

        Schema:{schema}

        Question:{question}

        SQL Query:
    """

    prompt=ChatPromptTemplate.from_messages([
        ("system","""
            You are an expert SQL Query generator.
            Generate only valid MySQL SQL Queries.
            Do not include explainations or markdowns.
        """),
        ("human",template)
    ])

    return prompt|llm|StrOutputParser()

In [164]:
# Test generation

sql_chain=write_sql_query(llm)

response=sql_chain.invoke({
    "schema":get_schema(),
    "question":"Show all tracks by AC/DC"
})

print(response)

SELECT t.`Name` AS TrackName, a.`Title` AS AlbumTitle, ar.`Name` AS ArtistName
FROM `track` t
JOIN `album` a ON t.`AlbumId` = a.`AlbumId`
JOIN `artist` ar ON a.`ArtistId` = ar.`ArtistId`
WHERE ar.`Name` = 'AC/DC';


In [165]:
result=run_query(response)

print(result)

Query being executed:SELECT t.`Name` AS TrackName, a.`Title` AS AlbumTitle, ar.`Name` AS ArtistName
FROM `track` t
JOIN `album` a ON t.`AlbumId` = a.`AlbumId`
JOIN `artist` ar ON a.`ArtistId` = ar.`ArtistId`
WHERE ar.`Name` = 'AC/DC';
[('For Those About To Rock (We Salute You)', 'For Those About To Rock We Salute You', 'AC/DC'), ('Put The Finger On You', 'For Those About To Rock We Salute You', 'AC/DC'), ("Let's Get It Up", 'For Those About To Rock We Salute You', 'AC/DC'), ('Inject The Venom', 'For Those About To Rock We Salute You', 'AC/DC'), ('Snowballed', 'For Those About To Rock We Salute You', 'AC/DC'), ('Evil Walks', 'For Those About To Rock We Salute You', 'AC/DC'), ('C.O.D.', 'For Those About To Rock We Salute You', 'AC/DC'), ('Breaking The Rules', 'For Those About To Rock We Salute You', 'AC/DC'), ('Night Of The Long Knives', 'For Those About To Rock We Salute You', 'AC/DC'), ('Spellbound', 'For Those About To Rock We Salute You', 'AC/DC'), ('Go Down', 'Let There Be Rock', 'A

### SQL Validator

In [166]:
import sqlglot
from sqlglot import exp

In [167]:
def validate_sql(query):

    try:
        parsed=sqlglot.parse(query,read="mysql")

        if len(parsed)!=1:
            return {
                "valid":False,
                "operation":None,
                "message":"Only one SQL Statement is allowed"
            }

        statement=parsed[0]

        if isinstance(statement,exp.Select):
            operation="SELECT"

        elif isinstance(statement,exp.Insert):
            operation="INSERT"

        elif isinstance(statement,exp.Update):
            operation="UPDATE"

        elif isinstance(statement,exp.Delete):
            operation="DELETE"

        elif isinstance(statement,exp.Alter):
            operation="ALTER"

        elif isinstance(statement,exp.Drop):
            operation="ALTER"

        elif isinstance(statement,exp.Create):
            operation="CREATE"

        else:
            return {
                "valid":False,
                "operation":None,
                "message":"Unsupported SQL operation"
            }

        return {
            "valid":True,
            "operation":operation,
            "message":f"{operation} query validated"
        }
    except Exception as e:
        return {
            "valid":False,
            "operation":None,
            "message":str(e)
        }


In [168]:
print(validate_sql("SELECT * FROM Track"))
print(validate_sql("DROP TABLE Track"))
print(validate_sql("DELETE FROM Track"))
print(validate_sql("UPDATE Track SET Name = 'Test'"))

{'valid': True, 'operation': 'SELECT', 'message': 'SELECT query validated'}
{'valid': True, 'operation': 'ALTER', 'message': 'ALTER query validated'}
{'valid': True, 'operation': 'DELETE', 'message': 'DELETE query validated'}
{'valid': True, 'operation': 'UPDATE', 'message': 'UPDATE query validated'}


### SQL Approval

In [169]:
def check_approval(operation):

    if operation=="SELECT":
        return False

    return True

In [170]:
operations=["DELETE","SELECT","INSERT","ALTER","UPDATE"]

for operation in operations:

    approval=check_approval(operation)

    if approval:
        print(operation,"-> Human approval required")

    else:
        print(operation,"->No approval required")

DELETE -> Human approval required
SELECT ->No approval required
INSERT -> Human approval required
ALTER -> Human approval required
UPDATE -> Human approval required


In [171]:
def prepare_sql(query):

    validation=validate_sql(query)

    if not validation['valid']:
        return {
            "status":"rejected",
            "query":query,
            "message":validation['message']
        }

    operation=validation['operation']

    approval_required=check_approval(operation)

    if approval_required:
        return {
            "status":"approval_required",
            "query":query,
            "operation":operation,
            "message":"Human Approval Required"
        }

    return {
        "status":"ready",
        "query":query,
        "operation":operation,
        "message":"Query is ready for execution"
    }


In [172]:
print(prepare_sql("SELECT * FROM Track LIMIT 5"))

print(prepare_sql(
    "UPDATE Artist SET Name = 'Updated Artist' WHERE ArtistId = 1"
))

print(prepare_sql("DROP TABLE Track"))

{'status': 'ready', 'query': 'SELECT * FROM Track LIMIT 5', 'operation': 'SELECT', 'message': 'Query is ready for execution'}
{'status': 'approval_required', 'query': "UPDATE Artist SET Name = 'Updated Artist' WHERE ArtistId = 1", 'operation': 'UPDATE', 'message': 'Human Approval Required'}
{'status': 'approval_required', 'query': 'DROP TABLE Track', 'operation': 'ALTER', 'message': 'Human Approval Required'}


### Agent

In [173]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Optional

In [174]:
class AgentState(TypedDict):

    question: str
    schema: str
    sql_query: str
    operation: Optional[str]
    status: str
    approval_required: bool
    result: Optional[str]
    answer:Optional[str]

### Response Agent

In [175]:
def response_chain(llm):

    prompt=ChatPromptTemplate.from_messages([
        ("system","""

        You are a helpful database assistant.
        Answer the user's question using only the SQL query results.
        Keep the answer simple and clear.
        If the results are empty, tell the user that no records were found.
        Do not make up any information.
        """),
        ("human", """
        User Question: {question}

        SQL Query: {sql_query}

        SQL Results: {result}

        Provide a natural language answer:


        """)
    ])

    return prompt|llm|StrOutputParser()

answer_chain=response_chain(llm)

In [176]:
def response_node(state:AgentState):

    answer=answer_chain.invoke({
        "question":state["question"],
        "sql_query":state["sql_query"],
        "result":state["result"]
    })

    return {
        "answer":answer
    }

In [177]:
# SQL Generator Node

def generate_sql_node(state:AgentState):

    response=sql_chain.invoke({
        "schema":state["schema"],
        "question":state["question"]
    })

    return {
        "sql_query":response
    }

In [178]:
test_state = {
    "question":"Show me all tracks by AC/DC",
    "schema":get_schema(),
    "sql_query":"",
    "operation":None,
    "status":"started",
    "approval_required":False,
    "result":None
}

output=generate_sql_node(test_state)
test_state.update(output)
print(output)

{'sql_query': "SELECT t.TrackId, t.Name AS TrackName, a.Title AS AlbumTitle\nFROM track t\nJOIN album a ON t.AlbumId = a.AlbumId\nJOIN artist ar ON a.ArtistId = ar.ArtistId\nWHERE ar.Name = 'AC/DC';"}


In [179]:
print(test_state["sql_query"])

SELECT t.TrackId, t.Name AS TrackName, a.Title AS AlbumTitle
FROM track t
JOIN album a ON t.AlbumId = a.AlbumId
JOIN artist ar ON a.ArtistId = ar.ArtistId
WHERE ar.Name = 'AC/DC';


In [180]:
def validate_sql_node(state:AgentState):

    decision=prepare_sql(state["sql_query"])

    return {
        "operation":decision.get("operation"),
        "status":decision.get("status"),
        "approval_required":decision["status"]=="approval_required"
    }

In [181]:
validation_output=validate_sql_node(test_state)

print(validation_output)

{'operation': 'SELECT', 'status': 'ready', 'approval_required': False}


### Router Function

In [182]:
from langgraph.types import interrupt , Command
from langgraph.checkpoint.memory import MemorySaver

In [183]:
def route_after_validation(state:AgentState):

    if state["status"]=="rejected":
        return "reject"

    elif state["approval_required"]:
        return "approval"

    else:
        return "execute"

In [184]:
test_state.update(validation_output)

print(route_after_validation(test_state))

execute


In [185]:
def approval_node(state:AgentState):

    decision=interrupt({
        "message":"Human approval required",
        "operation":state["operation"],
        "sql_query":state["sql_query"]
    })

    if decision=="approve":
        return {
            "status":"approved"
        }

    else:
        return {
            "status":"rejected"
        }

In [186]:
def route_after_approval(state:AgentState):

    if state['status']=="approved":
        return "execute"

    else:
        return "reject"

In [187]:
def execution_node(state:AgentState):
    return {
        "status":"execution_pending"
    }



def reject_node(state:AgentState):
    return {
        "status":"rejected"
    }

### Graph

In [188]:
graph=StateGraph(AgentState)

In [189]:
# Execution Node 

def execute_node(state:AgentState):

    result=run_query(state['sql_query'])

    return {
        "result":str(result),
        "status":"completed"
    }

In [190]:
graph.add_node("generate_sql",generate_sql_node)
graph.add_node("validate_sql",validate_sql_node)
graph.add_node("execute",execute_node)
graph.add_node("approval",approval_node)
graph.add_node("reject",reject_node)
graph.add_node("response",response_node)

In [191]:
graph.add_edge(START,"generate_sql")
graph.add_edge("generate_sql","validate_sql")

# Validation Routing
graph.add_conditional_edges(
    "validate_sql",
    route_after_validation,
    {
        "execute":"execute",
        "approval":"approval",
        "reject":"reject"
    }
)

# Approval Routing
graph.add_conditional_edges(
    "approval",
    route_after_approval,
    {
        "execute":"execute",
        "reject":"reject"
    }
)

In [192]:
graph.add_edge("execute","response")
graph.add_edge("response",END)

graph.add_edge("reject",END)

In [193]:
memory=MemorySaver()

In [194]:
app=graph.compile(checkpointer=memory)

In [195]:
# Testing 

config={
    "configurable":{
        "thread_id":"querymind-hitl-test"
    }
}

result=app.invoke({
    "question":"Change the name of the artist with ArtistID 1 to AC/DC.",
    "schema":get_schema(),
    "sql_query":"",
    "operation":None,
    "status":"started",
    "approval_required":False,
    "result":None
    },config=config)

print(result)

{'question': 'Change the name of the artist with ArtistID 1 to AC/DC.', 'schema': '\nCREATE TABLE album (\n\t`AlbumId` BIGINT NOT NULL AUTO_INCREMENT, \n\t`Title` VARCHAR(160) NOT NULL, \n\t`ArtistId` BIGINT NOT NULL, \n\tPRIMARY KEY (`AlbumId`), \n\tCONSTRAINT album_ibfk_1 FOREIGN KEY(`ArtistId`) REFERENCES artist (`ArtistId`)\n)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci\n\n\nCREATE TABLE artist (\n\t`ArtistId` BIGINT NOT NULL AUTO_INCREMENT, \n\t`Name` VARCHAR(120), \n\tPRIMARY KEY (`ArtistId`)\n)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci\n\n\nCREATE TABLE customer (\n\t`CustomerId` BIGINT NOT NULL AUTO_INCREMENT, \n\t`FirstName` VARCHAR(40) NOT NULL, \n\t`LastName` VARCHAR(20) NOT NULL, \n\t`Company` VARCHAR(80), \n\t`Address` VARCHAR(70), \n\t`City` VARCHAR(40), \n\t`State` VARCHAR(40), \n\t`Country` VARCHAR(40), \n\t`PostalCode` VARCHAR(10), \n\t`Phone` VARCHAR(24), \n\t`Fax` VARCHAR(24), \n\t`Email` VARCHAR(60) NOT NULL, \n\t`SupportRep

In [196]:
result=app.invoke(
    Command(resume="approve"),
    config=config
)

print(result)

Query being executed:UPDATE artist SET Name = 'AC/DC' WHERE ArtistId = 1;
{'question': 'Change the name of the artist with ArtistID 1 to AC/DC.', 'schema': '\nCREATE TABLE album (\n\t`AlbumId` BIGINT NOT NULL AUTO_INCREMENT, \n\t`Title` VARCHAR(160) NOT NULL, \n\t`ArtistId` BIGINT NOT NULL, \n\tPRIMARY KEY (`AlbumId`), \n\tCONSTRAINT album_ibfk_1 FOREIGN KEY(`ArtistId`) REFERENCES artist (`ArtistId`)\n)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci\n\n\nCREATE TABLE artist (\n\t`ArtistId` BIGINT NOT NULL AUTO_INCREMENT, \n\t`Name` VARCHAR(120), \n\tPRIMARY KEY (`ArtistId`)\n)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci\n\n\nCREATE TABLE customer (\n\t`CustomerId` BIGINT NOT NULL AUTO_INCREMENT, \n\t`FirstName` VARCHAR(40) NOT NULL, \n\t`LastName` VARCHAR(20) NOT NULL, \n\t`Company` VARCHAR(80), \n\t`Address` VARCHAR(70), \n\t`City` VARCHAR(40), \n\t`State` VARCHAR(40), \n\t`Country` VARCHAR(40), \n\t`PostalCode` VARCHAR(10), \n\t`Phone` VARCHAR(24)

In [197]:
db.run("""
SELECT ArtistId, Name
FROM artist
WHERE ArtistId = 1;
""")

"[(1, 'AC/DC')]"

In [199]:
config={
    "configurable":{
        "thread_id":"querymind-select-test"
    }
}

In [200]:
result = app.invoke({
    "question": "Show all tracks by AC/DC",
    "schema": get_schema(),
    "sql_query": "",
    "operation": None,
    "status": "started",
    "approval_required": False,
    "result": None,
    "answer": None
}, config=config)

Query being executed:SELECT t.TrackId, t.Name AS TrackName, a.Title AS AlbumTitle
FROM track t
JOIN album a ON t.AlbumId = a.AlbumId
JOIN artist ar ON a.ArtistId = ar.ArtistId
WHERE ar.Name = 'AC/DC';


In [201]:
print("Generated SQL:\n", result["sql_query"])
print("\nDatabase Result:\n", result["result"])
print("\nFinal Answer:\n", result["answer"])

Generated SQL:
 SELECT t.TrackId, t.Name AS TrackName, a.Title AS AlbumTitle
FROM track t
JOIN album a ON t.AlbumId = a.AlbumId
JOIN artist ar ON a.ArtistId = ar.ArtistId
WHERE ar.Name = 'AC/DC';

Database Result:
 [(1, 'For Those About To Rock (We Salute You)', 'For Those About To Rock We Salute You'), (6, 'Put The Finger On You', 'For Those About To Rock We Salute You'), (7, "Let's Get It Up", 'For Those About To Rock We Salute You'), (8, 'Inject The Venom', 'For Those About To Rock We Salute You'), (9, 'Snowballed', 'For Those About To Rock We Salute You'), (10, 'Evil Walks', 'For Those About To Rock We Salute You'), (11, 'C.O.D.', 'For Those About To Rock We Salute You'), (12, 'Breaking The Rules', 'For Those About To Rock We Salute You'), (13, 'Night Of The Long Knives', 'For Those About To Rock We Salute You'), (14, 'Spellbound', 'For Those About To Rock We Salute You'), (15, 'Go Down', 'Let There Be Rock'), (16, 'Dog Eat Dog', 'Let There Be Rock'), (17, 'Let There Be Rock', 'Let